In [1]:
# Imports 

import pandas as pd

In [2]:
# Read in restricted kidney waitlist data

df = pd.read_csv("../../data/restricted/kidney_waitlist_analytic.csv.gz", low_memory=False)

df.head()

,ON_DIALYSIS,A2A2B_ELIGIBILITY,GENDER,ABO,BMI_TCR,FUNC_STAT_TCR,INIT_STAT,INIT_CPRA,END_CPRA,REM_CD,...,PSTATUS,PTIME,TRR_ID_CODE,DONOR_ID,LISTING_CTR_CODE,outcome,event_adverse,event_transplant,censored,days_to_event
0,Y,NaN,F,B,31.63,2080.0,4099,0.0,0.0,8.0,...,NaN,NaN,NaN,NaN,13609,died,1,0,0,287.0
1,Y,NaN,M,A,30.04,2070.0,4099,0.0,NaN,24.0,...,NaN,NaN,NaN,NaN,6975,removed_administrative,0,0,0,2322.0
2,N,NaN,F,O,32.85,2070.0,4099,0.0,0.0,8.0,...,NaN,NaN,NaN,NaN,19716,died,1,0,0,604.0
3,Y,NaN,M,A,20.00,2090.0,4099,0.0,0.0,13.0,...,NaN,NaN,NaN,NaN,8587,removed_too_sick,1,0,0,909.0
4,Y,NaN,M,AB,23.30,2070.0,4010,0.0,0.0,13.0,...,NaN,NaN,NaN,NaN,18352,removed_too_sick,1,0,0,231.0


In [3]:
# Pre-drop processing

to_date = ['INIT_DATE', 'TX_DATE', 'END_DATE']
for value in to_date:
    df[value] = pd.to_datetime(df[value], errors='coerce')

# df['TRANSPLANTED'] = df['TX_DATE'].notna()

# calculate how long the patient waited
# patients with no transplant date get a blank
df['DAYS_TO_EVENT'] = (df['TX_DATE'] - df['INIT_DATE']).dt.days

bad_data = (df['TX_DATE'] < df['INIT_DATE']) | (df['END_DATE'] < df['INIT_DATE'])

df = df[~bad_data].copy()

In [4]:
# Remove columns related to post-transplant data

post_tsplt_cols = [
    'REM_CD', 'END_DATE', 'TX_DATE', 'COMPOSITE_DEATH_DATE',
    'END_STAT', 'END_CPRA', 'DAYSWAIT_CHRON', 'DAYSWAIT_ALLOC',
    'DON_TY', 'ORGAN', 'TRR_ID_CODE', 'DONOR_ID', 'PTIME', 'PSTATUS'
]

df.drop(columns=post_tsplt_cols, errors="ignore", inplace=True)

In [5]:
# Fill missing categorical values

df['MULTIORG'] = df['MULTIORG'].fillna('N')

string_not_dia_cols = df.select_dtypes(include=["object", 'str']).columns.difference(["DIALYSIS_DATE"])
df[string_not_dia_cols] = df[string_not_dia_cols].fillna("Unknown")

In [6]:
# Fill missing numeric values

cts_cols = ['BMI_TCR', 'INIT_CPRA']
df[cts_cols] = df[cts_cols].fillna(df[cts_cols].median())

cols = ["DIAG_KI", "FUNC_STAT_TCR", "A2A2B_ELIGIBILITY"]
df[cols] = df[cols].fillna(-1).astype(str)

In [7]:
# Winscorize BMI_TCR and clip outliers

q1, q3 = df["BMI_TCR"].quantile([0.25, 0.75])
iqr = q3 - q1

df["BMI_TCR"] = df["BMI_TCR"].clip(
    lower=q1 - 1.5 * iqr,
    upper=q3 + 1.5 * iqr
)

In [8]:
date_cols = ["INIT_DATE", "DIALYSIS_DATE"]
df[date_cols] = df[date_cols].apply(pd.to_datetime)

df["DIALYSIS_DURATION"] = (
    df["INIT_DATE"] - df["DIALYSIS_DATE"]
).dt.days.fillna(0)

df["DIALYSIS_AFTER_LISTING"] = (
    df["DIALYSIS_DURATION"] < 0
).astype("int8")

df["DIALYSIS_DURATION"] = df["DIALYSIS_DURATION"].clip(lower=0)

df.drop(columns="DIALYSIS_DATE", inplace=True)

In [9]:
df.isna().sum()

ON_DIALYSIS                    0
A2A2B_ELIGIBILITY              0
GENDER                         0
ABO                            0
BMI_TCR                        0
FUNC_STAT_TCR                  0
INIT_STAT                      0
INIT_CPRA                      0
INIT_AGE                       0
INIT_DATE                      0
ETHCAT                         0
PT_CODE                        0
REGION                         0
WL_ID_CODE                     0
PREV_TX                        0
DIAG_KI                        0
MULTIORG                       0
LISTING_CTR_CODE               0
outcome                        0
event_adverse                  0
event_transplant               0
censored                       0
days_to_event                  0
DAYS_TO_EVENT             264048
DIALYSIS_DURATION              0
DIALYSIS_AFTER_LISTING         0
dtype: int64